# Feature Scaling


Some Machine Learning models care a lot about the scale of the input features. If one variable is measured in fractions while another is measured in the hundreds or thousands, a distance-based model can treat the large-scale feature as much more important even when that is not what we want.

In this notebook, we use the wine dataset to compare a K-nearest neighbors classifier before and after feature scaling.


The core idea is simple:

```mermaid
flowchart LR
    A[Raw features on<br>different scales] --> B[KNN computes<br>distances]
    B --> C[Large-scale<br>features dominate]
    A --> D[StandardScaler]
    D --> E[Features on a<br>comparable scale]
    E --> F[More balanced<br>distance calculations]
```


## Load the Dataset


We use the wine dataset built into scikit-learn. It is a multi-class classification problem with only numerical features, which makes it a good example for scaling.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df.head()

Before training, it helps to inspect the feature scales. KNN uses distances between points, so a feature with values in the hundreds can easily dominate a feature with values near zero.


In [ ]:
scale_overview = df.drop(columns="target").agg(["mean", "std", "min", "max"]).T
scale_overview

Notice how different the ranges are. For example, features such as `proline` have much larger values than features such as `nonflavanoid_phenols`. Without scaling, that difference heavily influences distance-based models.


## Train a Baseline KNN Model


We first train a KNN classifier on the raw features. Because the dataset has multiple classes, we use `stratify=y` so the train and test split keep a similar class distribution.


In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y,
)

knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)

y_pred_raw = knn_raw.predict(X_test)
raw_accuracy = accuracy_score(y_test, y_pred_raw)
print(f"Accuracy without scaling: {raw_accuracy:.3f}")

That baseline gives us a reference point. Next, we standardize the features so each column has roughly mean `0` and standard deviation `1` on the training data.


## Apply Standardization


`StandardScaler` learns the mean and standard deviation from the training data only. We then use the same transformation on the test set. This is important because fitting the scaler on all data would leak information from the test set into the training process.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
scaled_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
scaled_train_df.agg(["mean", "std"]).T.head(10)

The table is helpful, but a visual comparison makes the scale problem even more obvious. The boxplots below compare a few features before and after scaling.


In [ ]:
feature_subset = ["proline", "magnesium", "nonflavanoid_phenols"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df[feature_subset].boxplot(ax=axes[0])
axes[0].set_title("Before scaling")
axes[0].tick_params(axis="x", rotation=45)

scaled_train_df[feature_subset].boxplot(ax=axes[1])
axes[1].set_title("After scaling")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

The scaled training features are now centered near `0` and have similar spread. That gives distance-based algorithms a fairer view of all features.


## Train the Model on Scaled Features


In [ ]:
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = knn_scaled.predict(X_test_scaled)
scaled_accuracy = accuracy_score(y_test, y_pred_scaled)
print(f"Accuracy with scaling: {scaled_accuracy:.3f}")

In practice, preprocessing and model training are often combined in a `Pipeline`. That keeps the workflow cleaner and makes it easier to reuse the same steps during cross-validation or deployment.

In [ ]:
knn_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]
)

knn_pipeline.fit(X_train, y_train)
pipeline_accuracy = accuracy_score(y_test, knn_pipeline.predict(X_test))
print(f"Accuracy with a pipeline: {pipeline_accuracy:.3f}")

## Compare the Results


In [ ]:
pd.DataFrame(
    {
        "model": [
            "KNN without scaling",
            "KNN with StandardScaler",
            "KNN pipeline",
        ],
        "accuracy": [raw_accuracy, scaled_accuracy, pipeline_accuracy],
    }
)

The improvement happens because KNN depends directly on distances. When features are brought onto a comparable scale, no single high-magnitude column dominates the neighborhood calculation.

Feature scaling is especially important for models such as KNN, support vector machines, and logistic regression. Tree-based models such as decision trees and random forests are usually much less sensitive to scaling.
